In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

os.chdir("..")

load_dotenv(dotenv_path="config/.env")

API_KEY = os.getenv("OPENROUTER_API_KEY")

In [5]:
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=API_KEY,
)

In [ ]:
from pydantic import Field
from pydantic import BaseModel
from typing import Literal
from aether.llm.base import BaseLLM
from aether.llm.structured import StructuredLLM

class CodeReview(BaseModel):
    issues: list[str] = Field(description="발견된 문제점들")
    suggestions: list[str] = Field(description="개선 제안")
    severity: Literal["low", "medium", "high"] = Field(description="심각도")


structured_llm = StructuredLLM(
    client=client,
    schema=CodeReview,
    model="deepseek/deepseek-v3.2-exp",
    temperature=0.0,
    max_tokens=5000,
    max_retries=3,
)

messages = [
    {
        "role": "user",
        "content": """
        다음 코드를 리뷰해주세요:
        
        def divide(a, b):
            return a / b
        """
    }
]

result = structured_llm.invoke(messages)

In [16]:
llm = BaseLLM(model="deepseek/deepseek-v3.2-exp", client=client)

In [18]:
result = llm.invoke(messages)

In [20]:
structured_llm = llm.with_structured_output(CodeReview)

In [21]:
structured_result = structured_llm.invoke(messages)

In [23]:
structured_result.model_dump()

{'issues': ['0으로 나누는 경우 ZeroDivisionError 예외가 발생할 수 있음',
  '입력값이 숫자 타입인지 검증하지 않음'],
 'suggestions': ['b가 0인 경우를 확인하고 적절히 처리하도록 수정', '입력 매개변수의 타입을 검증하는 로직 추가'],
 'severity': 'high'}

In [6]:
from aether.llm.prompt import PromptLoader

prompt_loader = PromptLoader(base_path = "aether/prompts")

In [23]:
SYSTEM_PROMPT = prompt_loader.load(
    "greeting.md", 
    name="Alice", 
    role="developer", 
    project="Aether"
    )

In [24]:
print(SYSTEM_PROMPT)

Hello Alice!

Welcome to the Aether project. You are logged in as developer.
